In [10]:
import json
from dotenv import load_dotenv
load_dotenv('../.env')

import litellm
from typing import List, Literal
from pypdf import PdfReader, PdfWriter
from pathlib import Path
import base64
import os
import tempfile
from pydantic import BaseModel, Field


In [11]:
DICTFILE = '../inputs/carolinian_full.pdf'
PAGES = [494, 1229]


In [12]:
class TextItem(BaseModel):
    text: str = Field(..., description="The text content, without any dialect marker.")
    style: str = Field(..., description="'bold' if this is an English concept heading, 'italic' if this is a Carolinian headword.")
    dialect: str = Field("", description="Dialect marker if present in parentheses immediately after the italic word, e.g. TAN, LN, EL, S. Empty string if none.")


class Page(BaseModel):
    items: list[TextItem]
    number: int
    file: str

    @classmethod
    def load_pages(cls, filepath: str = "finderlist_pages.jsonl") -> list["Page"]:
        with open(filepath, "r") as f:
            return [cls.model_validate_json(line) for line in f]

    @staticmethod
    def save_pages(pages: list["Page"], filepath: str = "finderlist_pages.jsonl") -> None:
        with open(filepath, "w") as f:
            for page in pages:
                f.write(page.model_dump_json() + "\n")


In [13]:
response_schema = {
    "type": "array",
    "items": TextItem.model_json_schema()
}


In [14]:
def is_already_extracted(page_number: int, pages: List[Page]) -> bool:
    return any(page.number == page_number for page in pages)


In [15]:
def extract_entries(pdf_path, page_number) -> tuple:
    """Returns (Page, cost_usd)."""
    reader = PdfReader(pdf_path)
    page = reader.pages[page_number]
    text = page.extract_text()
    writer = PdfWriter()
    writer.add_page(page)
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmpfile:
        writer.write(tmpfile)
        tmpfile_path = tmpfile.name

    pdf_bytes = Path(tmpfile_path).read_bytes()
    encoded_data = base64.b64encode(pdf_bytes).decode("utf-8")

    prompt = (
        "<page_text>\n" + text + "\n</page_text>\n\n"
        "This is a page from the English-Carolinian finder list of a bilingual dictionary. "
        "Extract all text items in reading order. "
        "Bold words are English concept headings (style='bold'). "
        "Italic words are Carolinian headwords (style='italic'). "
        "After each italic Carolinian headword, there may be a dialect marker in parentheses "
        "in small caps (TAN, LN, EL, S) — capture this in the dialect field. "
        "All other normal (non-bold, non-italic) text is English description — do NOT include it. "
        "Return only bold and italic items in order as JSON."
    )

    model = "gemini/gemini-3-flash-preview"
    response = litellm.completion(
        model=model,
        response_format={"type": "json_object", "response_schema": response_schema},
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "file", "file": {"file_data": f"data:application/pdf;base64,{encoded_data}"}},
            ],
        }],
    )

    os.remove(tmpfile_path)

    cost = litellm.completion_cost(response)

    import re
    content = response['choices'][0]['message']['content']
    # Fix invalid unicode escapes before parsing
    content = re.sub(r'\\u(?![0-9a-fA-F]{4})', r'\\\\u', content)
    result = Page(
        items=[TextItem(**e) for e in json.loads(content)],
        number=page_number,
        file=pdf_path
    )
    return result, cost


In [16]:
try:
    pages = Page.load_pages()
    print(f"Loaded {len(pages)} already-extracted pages")
except FileNotFoundError:
    pages = []
    print("No existing pages found, starting fresh")


Loaded 726 already-extracted pages


In [17]:
# Pages to overwrite (0-indexed)
OVERWRITE_PAGES = [764, 888, 557, 894, 796, 889, 989, 631, 653]

pages = [p for p in pages if p.number not in OVERWRITE_PAGES]
Page.save_pages(pages)
print(f"Removed {len(OVERWRITE_PAGES)} pages from cache, will re-extract them.")


Removed 9 pages from cache, will re-extract them.


In [18]:
from tqdm import tqdm
import time

total_cost = 0.0

for page_number in tqdm(range(*PAGES)):
    if is_already_extracted(page_number, pages):
        print(f"Page {page_number} already extracted, skipping.")
        continue

    try:
        page, cost = extract_entries(DICTFILE, page_number)
    except json.JSONDecodeError as e:
        print(f"JSON parse error on page {page_number}: {e}, skipping.")
        continue
    except litellm.exceptions.RateLimitError:
        print(f"Rate limit on page {page_number}, waiting 60s...")
        time.sleep(60)
        page, cost = extract_entries(DICTFILE, page_number)

    total_cost += cost
    print(f"Page {page_number}: {len(page.items)} items | cost=${cost:.4f} | running total=${total_cost:.4f}")
    pages.append(page)
    Page.save_pages(pages)

print(f"\nDone. Total cost: ${total_cost:.4f}")


  0%|                                                   | 0/735 [00:00<?, ?it/s]

Page 494 already extracted, skipping.
Page 495 already extracted, skipping.
Page 496 already extracted, skipping.
Page 497 already extracted, skipping.
Page 498 already extracted, skipping.
Page 499 already extracted, skipping.
Page 500 already extracted, skipping.
Page 501 already extracted, skipping.
Page 502 already extracted, skipping.
Page 503 already extracted, skipping.
Page 504 already extracted, skipping.
Page 505 already extracted, skipping.
Page 506 already extracted, skipping.
Page 507 already extracted, skipping.
Page 508 already extracted, skipping.
Page 509 already extracted, skipping.
Page 510 already extracted, skipping.
Page 511 already extracted, skipping.
Page 512 already extracted, skipping.
Page 513 already extracted, skipping.
Page 514 already extracted, skipping.
Page 515 already extracted, skipping.
Page 516 already extracted, skipping.
Page 517 already extracted, skipping.
Page 518 already extracted, skipping.
Page 519 already extracted, skipping.
Page 520 alr

  9%|███▌                                      | 63/735 [01:01<10:59,  1.02it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



RateLimitError: litellm.RateLimitError: litellm.RateLimitError: GeminiException - {
  "error": {
    "code": 429,
    "message": "Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ",
    "status": "RESOURCE_EXHAUSTED"
  }
}


## Post-processing: group italic Carolinian headwords under bold English concepts

In [ ]:
import csv

pages_sorted = sorted(pages, key=lambda p: p.number)

# Walk all items in page order, group italic items under preceding bold
entries = []
current_english = None

for page in pages_sorted:
    for item in page.items:
        if item.style == "bold":
            current_english = item.text.strip()
        elif item.style == "italic" and current_english:
            entries.append({
                "english_word": current_english,
                "carolinian_headword": item.text.strip(),
                "dialect": item.dialect.strip(),
                "page_number": page.number,
            })

print(f"Total entries: {len(entries)}")
print(f"Sample:")
for e in entries[:5]:
    print(e)


In [ ]:
with open("finderlist.tsv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["english_word", "carolinian_headword", "dialect", "page_number"], delimiter="\t")
    writer.writeheader()
    writer.writerows(entries)

print("Saved → finderlist.tsv")
